# Lunar hazard rock detection: training runs

Runs one experiment per session: set `runKey` in the config cell, then **Runtime > Run all**.

Before the first run:
1. **Runtime > Change runtime type > T4 GPU**
2. Put the project's `scripts/` folder in Google Drive at `MyDrive/rover-navigation-hazard-detection/scripts/`
3. Add your Kaggle API token (starts with `KGAT_`) in the 🔑 Secrets panel (left sidebar) as `KAGGLE_API_TOKEN`, with notebook access on

In [ ]:
# config: change runKey for each experiment
runKey = "baseline"     # baseline | contrast | blur | noise | lighting | all
seed = 0                # change only for repeat runs
modelName = "yolov8n.pt"
epochs = 50
imgSize = 640
batchSize = 32

runNames = {"baseline": "Baseline (no augmentation)", "contrast": "Contrast and exposure",
            "blur": "Gaussian blur", "noise": "Sensor noise",
            "lighting": "Directional lighting", "all": "All combined"}
assert runKey in runNames, f"unknown runKey {runKey}"

In [ ]:
# gpu check, should show a tesla t4
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# drive holds the scripts and keeps results if colab disconnects
from google.colab import drive
drive.mount("/content/drive")
driveDir = "/content/drive/MyDrive/rover-navigation-hazard-detection"
!mkdir -p {driveDir}/runs {driveDir}/results
!mkdir -p /content/project && cp -r {driveDir}/scripts /content/project/
%cd /content/project
!ls scripts

In [ ]:
!pip -q install -U ultralytics albumentations "kaggle>=1.8"


In [ ]:
# dataset straight from kaggle (faster than uploading 4 gb)
import os
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN").strip()   # strip stray newline from paste
if not os.path.exists("data/raw/lunar/bounding_boxes.csv"):
    !kaggle datasets download -d romainpessia/artificial-lunar-rocky-landscape-dataset -p data/raw/lunar --unzip -q
!ls data/raw/lunar


In [ ]:
# same scripts + same seed as the mac, so the split matches exactly
!python scripts/convertLabels.py
!python scripts/splitData.py

In [ ]:
# augmentation runs: build the augmented training set first
# (augment.py comes in a later step; baseline skips this)
if runKey != "baseline":
    !python scripts/augment.py --aug {runKey} --seed {seed}
    trainYaml = f"data_{runKey}.yaml"
else:
    trainYaml = "data.yaml"
print("training with", trainYaml)

In [ ]:
from ultralytics import YOLO

runName = f"{runKey}_s{seed}"
model = YOLO(modelName)
model.train(data=trainYaml, epochs=epochs, imgsz=imgSize, batch=batchSize, seed=seed,
            deterministic=True, project=f"{driveDir}/runs", name=runName, exist_ok=True,
            patience=15, workers=2)

In [ ]:
# score the best weights on renders (val) and on the real photos (test)
import csv
bestWeights = f"{driveDir}/runs/{runName}/weights/best.pt"
best = YOLO(bestWeights)
valStats = best.val(data="data.yaml", split="val", imgsz=imgSize, verbose=False)
realStats = best.val(data="data.yaml", split="test", imgsz=imgSize, verbose=False,
                     project=f"{driveDir}/runs/{runName}", name="realTest", exist_ok=True)

row = {"key": runKey, "name": runNames[runKey], "seed": seed,
       "val_map50": round(valStats.box.map50, 4),
       "real_map50": round(realStats.box.map50, 4),
       "real_map5095": round(realStats.box.map, 4),
       "ms": round(realStats.speed["inference"], 2)}
print(row)

# one shared log on drive, same format update_site.py reads
logPath = f"{driveDir}/results/experiments.csv"
isNew = not os.path.exists(logPath)
with open(logPath, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(row))
    if isNew:
        w.writeheader()
    w.writerow(row)
print("logged to", logPath)

Results land in Drive at `rover-navigation-hazard-detection/runs/<runKey>_s<seed>/` (weights, curves, `realTest/` plots) and the shared log `rover-navigation-hazard-detection/results/experiments.csv`.

The ms number is measured on the T4. When you compare runs, they all need the same GPU type.